In [1]:
import cogent3
from cogent3 import get_app
from cogent3 import load_aligned_seqs
from cogent3.maths.matrix_exponential_integration import expected_number_subs

# Calculating Intergenic Ancestral Repeat Q

In [2]:
folder_in = 'igar/'

sequence = 'homo_sapiens-22-36038797-36056389.fa'

omit_degs_noncds = get_app("omit_degenerates", moltype="dna")

aln_igar = load_aligned_seqs(filename = folder_in + sequence, moltype='dna')
aln_igar = omit_degs_noncds(aln_igar)

GN_subsmodel = get_app("model", "GN", time_het="max", lf_args={"discrete_edges": ["Gorilla"]}, optimise_motif_probs=True, show_progress=True)
result_IGAR = GN_subsmodel(aln_igar)

humanQ_IGAR = result_IGAR.lf.get_rate_matrix_for_edge("Human", calibrated=False)

   0%|          |00:00<?

   0%|          |00:00<?

In [5]:
result_IGAR.lf

GN
log-likelihood = -17863.6784
number of free parameters = 39
======================================================================
edge          parent    length     A>C     A>G     A>T     C>A     C>G
----------------------------------------------------------------------
Human         root        0.01    1.32    2.81    0.52    0.90    0.36
Chimpanzee    root        0.01    1.42    5.21    2.39    3.25    1.08
Gorilla       root          NA      NA      NA      NA      NA      NA
----------------------------------------------------------------------

continued: 
===============================================
  C>T      G>A     G>C     G>T     T>A      T>C
-----------------------------------------------
 5.76     6.12    0.28    1.42    0.55     2.70
22.24    20.03    3.72    3.07    2.49    15.04
   NA       NA      NA      NA      NA       NA
-----------------------------------------------

============================
   A       C       G       T
----------------------------
0.27    0.23    0.25    0.25
----------------------------
=========================
motif    motif2    dpsubs
-------------------------
T        T           0.99
T        C           0.01
T        A           0.00
T        G           0.00
C        T           0.01
C        C           0.99
C        A           0.00
C        G           0.00
A        T           0.00
A        C           0.00
A        A           0.99
A        G           0.00
G        T           0.00
G        C           0.00
G        A           0.01
G        G           0.98
-------------------------

# Calculating cds ENS

In [3]:
#folder_in = paths.DATA_APES114 + 'cds/codon_aligned/'
folder_in = 'cds/'

sequence = 'ENSG00000133466.fa'

omit_degs_cds = get_app("omit_degenerates", moltype="dna", motif_length=3)

aln_cds = load_aligned_seqs(filename = folder_in + sequence, moltype='dna')
omit_degs_cds(aln_cds)

result_cds = GN_subsmodel(aln_cds)

cds_motif_probs = result_cds.lf.get_param_value("mprobs")
humanQ_cds = result_cds.lf.get_rate_matrix_for_edge("Human", calibrated=False)
humanENS_cds = expected_number_subs(cds_motif_probs, humanQ_cds, t=1.0)
#This is the ENS of cds if selection was the same as on IGAR. Instead of using the cds Q matrix humanQ_cds, I use the IGAR Q humanQ_IGAR
humanENS_cds_IGARQ = expected_number_subs(cds_motif_probs, humanQ_IGAR, t=1.0)


print("Human ENS cds: ")
print(humanENS_cds)
print("Human ENS cds with IGAR Q: ")
print(humanENS_cds_IGARQ)

#result_cds.lf

   0%|          |00:00<?

   0%|          |00:00<?

Human ENS cds: 
0.004810325939899154
Human ENS cds with IGAR Q: 
0.008025096693524934


# Calculating Intergenic ENS

In [7]:
#folder_in = paths.DATA_APES114 + 'cds/codon_aligned/'
folder_in = 'ig/'

sequence = 'homo_sapiens-22-35347992-35372172.fa'

aln_ig = load_aligned_seqs(filename = folder_in + sequence, moltype='dna')
aln_ig = omit_degs_noncds(aln_ig)

result_ig = GN_subsmodel(aln_ig)

ig_motif_probs = result_ig.lf.get_param_value("mprobs")
humanQ_ig = result_ig.lf.get_rate_matrix_for_edge("Human", calibrated=False)
humanENS_ig = expected_number_subs(ig_motif_probs, humanQ_ig, t=1.0)
#This is the ENS of cds if selection was the same as on IGAR. Instead of using the cds Q matrix humanQ_cds, I use the IGAR Q humanQ_IGAR
humanENS_ig_IGARQ = expected_number_subs(ig_motif_probs, humanQ_IGAR, t=1.0)


print("Human ENS IG: ")
print(humanENS_ig)
print("Human ENS IG with IGAR Q: ")
print(humanENS_ig_IGARQ)

   0%|          |00:00<?

   0%|          |00:00<?

Human ENS IG: 
0.006526356909947123
Human ENS IG with IGAR Q: 
0.007860543750877583


# Calculating cds constraint

In [4]:
cds_constraint = (humanENS_cds_IGARQ - humanENS_cds)/humanENS_cds_IGARQ
cds_constraint

np.float64(0.4005896597133371)

In [8]:
ig_constraint = (humanENS_ig_IGARQ - humanENS_ig)/humanENS_ig_IGARQ
ig_constraint

np.float64(0.16973213090780215)